# Redrob Candidate Ranking System
## Team Clover - Colab Sandbox Demo
This notebook demonstrates our end-to-end 3-stage neural ranking pipeline using 50 test candidates.
**Prerequisite:** Ensure you have uploaded `test_candidates.jsonl` to your Colab workspace.

In [ ]:
!pip install -q transformers torch sentence-transformers pandas

### 1. Imports & Setup

In [ ]:
import json
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import re


### 2. The Plausibility Filter (Fraud Detection)

In [ ]:
def is_honeypot(candidate):
    profile = candidate.get("profile", {})
    skills = candidate.get("skills", [])
    
    # Check 1: Expert proficiency with 0 duration
    expert_zero = sum(1 for s in skills if s.get("proficiency") == "expert" and s.get("duration_months", 0) == 0)
    if expert_zero >= 5:
        return True
        
    # Check 2: Time-traveling tech
    for s in skills:
        name = s.get("name", "").lower()
        dur = s.get("duration_months", 0)
        if name == "llama-2" and dur > 38:
            return True
            
    return False


### 3. Pipeline Execution (Bi-Encoder + Cross-Encoder)

In [ ]:
def run_pipeline(candidates_list):
    print(f"Ingested {len(candidates_list)} candidates.")
    
    # Stage 1: Filter honeypots
    valid = [c for c in candidates_list if not is_honeypot(c)]
    print(f"{len(valid)} passed plausibility checks.")
    
    # Stage 2: Bi-Encoder
    print("Loading Bi-Encoder (all-MiniLM-L6-v2)...")
    bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
    jd_text = "Senior AI Engineer Founding Team NLP ML Retrieval Ranking Vector Databases Embeddings"
    jd_emb = bi_encoder.encode([jd_text])[0]
    
    scores = []
    for c in valid:
        career = " ".join([j.get("description", "") for j in c.get("career_history", [])])
        if not career:
             continue
        c_emb = bi_encoder.encode([career])[0]
        sim = float(torch.nn.functional.cosine_similarity(torch.tensor(jd_emb).unsqueeze(0), torch.tensor(c_emb).unsqueeze(0))[0])
        
        yoe = c.get("profile", {}).get("years_of_experience", 0)
        if 6 <= yoe <= 8:
            sim *= 1.05
        elif yoe < 5:
            sim *= 0.72
            
        scores.append({"candidate_id": c["candidate_id"], "score": sim, "raw": c, "career": career})
        
    scores.sort(key=lambda x: x["score"], reverse=True)
    top_10 = scores[:10]
    
    # Stage 3: Cross-Encoder
    print("Running Cross-Encoder (ms-marco-MiniLM) on Top 10...")
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    pairs = [[jd_text, item["career"]] for item in top_10]
    cross_scores = cross_encoder.predict(pairs)
    
    for idx, item in enumerate(top_10):
        item["final_score"] = item["score"] * 0.6 + (float(cross_scores[idx]) * 0.4)
        
    top_10.sort(key=lambda x: x["final_score"], reverse=True)
    
    results = []
    for i, item in enumerate(top_10):
        c = item["raw"]
        title = c.get("profile", {}).get("current_title", "")
        yoe = c.get("profile", {}).get("years_of_experience", 0)
        results.append({"Rank": i+1, "Candidate ID": item["candidate_id"], "Title": title, "YoE": yoe, "Score": round(item["final_score"], 3)})
        
    return pd.DataFrame(results)


### 4. Load Data & Run

In [ ]:
try:
    with open("test_candidates.jsonl", "r", encoding="utf-8") as f:
        candidates = [json.loads(line) for line in f]
    df_results = run_pipeline(candidates)
    display(df_results)
except FileNotFoundError:
    print("Error: test_candidates.jsonl not found. Please upload it!")
